## 1. 서울시 공동주택 위경도 변환

### 1) 카카오맵 API 설정

In [ ]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

In [ ]:
# 카카오맵 API 키 불러오기: API 키는 환경변수로 관리
load_dotenv()

KAKAO_API_KEY_1 = os.getenv("KAKAO_API_KEY_1")
KAKAO_API_KEY_2 = os.getenv("KAKAO_API_KEY_2")

In [445]:
import requests

url = "https://dapi.kakao.com/v2/local/search/address.json"

# REST API 키: KAKAO_API_KEY_1
headers = {

    "Authorization": "KakaoAK KAKAO_API_KEY_1"

}

params = {
    "query": "서울특별시 강남구 테헤란로 212"
}

res = requests.get(url, headers=headers, params=params)
print(res.json())

{'documents': [{'address': {'address_name': '서울 강남구 역삼동 718-5', 'b_code': '1168010100', 'h_code': '1168065000', 'main_address_no': '718', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_h_name': '역삼2동', 'region_3depth_name': '역삼동', 'sub_address_no': '5', 'x': '127.039600248343', 'y': '37.5012767241426'}, 'address_name': '서울 강남구 테헤란로 212', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '서울 강남구 테헤란로 212', 'building_name': '멀티캠퍼스', 'main_building_no': '212', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_name': '역삼동', 'road_name': '테헤란로', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.039600248343', 'y': '37.5012767241426', 'zone_no': '06220'}, 'x': '127.039600248343', 'y': '37.5012767241426'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}


In [447]:
# 한국 부동산원 기본정보 데이터 불러오기
df_real_estate = pd.read_csv('data/한국부동산원_공동주택 단지 식별정보_기본정보_20250918.csv', encoding='utf-8')
df_real_estate.head()

,단지고유번호,필지고유번호,주소,단지명_공시가격,단지명_건축물대장,단지명_도로명주소,단지종류,동수,세대수,사용승인일
0,11110200000003,1111010100100010000,서울특별시 종로구 청운동 1,청운벽산빌리지,NaN,청운벽산빌리지,2,9,126,1988-11-11
1,11110200000002,1111010100100030000,서울특별시 종로구 청운동 3,인텔빌라BC동,인텔빌라,인텔빌라,2,2,18,1990-11-27
2,11110200000005,1111010100100030150,서울특별시 종로구 청운동 3-150,인텔빌라A동,인텔빌라,인텔빌라,2,1,8,1988-12-01
3,11110220072004,1111010100100040001,서울특별시 종로구 청운동 4-1,아델하우스,청운동 파라디아 아델하우스,청운동 파라디아 아델하우스,2,1,12,2007-07-25
4,11110220072005,1111010100100040003,서울특별시 종로구 청운동 4-3,GRACETUSCANII,투스칸2,투스칸2,2,1,4,2007-06-19


In [448]:
df_real_estate.info()

<class 'pandas.DataFrame'>
RangeIndex: 307407 entries, 0 to 307406
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   단지고유번호     307407 non-null  int64
 1   필지고유번호     307407 non-null  int64
 2   주소         307407 non-null  str  
 3   단지명_공시가격   307407 non-null  str  
 4   단지명_건축물대장  184205 non-null  str  
 5   단지명_도로명주소  184075 non-null  str  
 6   단지종류       307407 non-null  int64
 7   동수         307407 non-null  int64
 8   세대수        307407 non-null  int64
 9   사용승인일      307407 non-null  str  
dtypes: int64(5), str(5)
memory usage: 23.5 MB


In [450]:
# 필요한 컬럼: 주소, 단지명_공시가격, 단지종류, 세대수
df_real_estate = df_real_estate[['주소', '단지명_공시가격', '단지종류', '세대수']]

In [452]:
# 서울특별시 필터링
df_real_estate_seoul = df_real_estate[df_real_estate['주소'].str.contains('서울특별시', na=False)]
df_real_estate_seoul.head()

,주소,단지명_공시가격,단지종류,세대수
0,서울특별시 종로구 청운동 1,청운벽산빌리지,2,126
1,서울특별시 종로구 청운동 3,인텔빌라BC동,2,18
2,서울특별시 종로구 청운동 3-150,인텔빌라A동,2,8
3,서울특별시 종로구 청운동 4-1,아델하우스,2,12
4,서울특별시 종로구 청운동 4-3,GRACETUSCANII,2,4


In [453]:
df_real_estate_seoul.info()

<class 'pandas.DataFrame'>
RangeIndex: 106097 entries, 0 to 106096
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   주소        106097 non-null  str  
 1   단지명_공시가격  106097 non-null  str  
 2   단지종류      106097 non-null  int64
 3   세대수       106097 non-null  int64
dtypes: int64(2), str(2)
memory usage: 3.2 MB


In [454]:
# 구 추출
df_real_estate_seoul['구'] = df_real_estate_seoul['주소'].str.extract(r'서울특별시\s+([^ ]+)')
df_real_estate_seoul.head()

,주소,단지명_공시가격,단지종류,세대수,구
0,서울특별시 종로구 청운동 1,청운벽산빌리지,2,126,종로구
1,서울특별시 종로구 청운동 3,인텔빌라BC동,2,18,종로구
2,서울특별시 종로구 청운동 3-150,인텔빌라A동,2,8,종로구
3,서울특별시 종로구 청운동 4-1,아델하우스,2,12,종로구
4,서울특별시 종로구 청운동 4-3,GRACETUSCANII,2,4,종로구


In [455]:
# 카카오맵 API로 위도, 경도 정보 가져오기
params = {
    "query": df_real_estate_seoul['주소'].iloc[0]
}
res = requests.get(url, headers=headers, params=params)
print(res.json())
# 위도, 경도 정보 추출
if res.status_code == 200 and res.json().get('documents'):
    latitude = res.json()['documents'][0]['y']
    longitude = res.json()['documents'][0]['x']
    print(f"위도: {latitude}, 경도: {longitude}")
# 위도, 경도 정보 가져오기 함수 정의
def get_lat_long(address):
    params = {
        "query": address
    }
    res = requests.get(url, headers=headers, params=params)
    if res.status_code == 200 and res.json().get('documents'):
        latitude = res.json()['documents'][0]['y']
        longitude = res.json()['documents'][0]['x']
        return latitude, longitude
    else:
        return None, None
# 위도, 경도 정보 가져오기 함수 테스트
test_address = df_real_estate_seoul['주소'].iloc[0]
lat, long = get_lat_long(test_address)
print(f"주소: {test_address}, 위도: {lat}, 경도: {long}")
# 위도, 경도 정보 가져오기 함수 적용
df_real_estate_seoul['위도'], df_real_estate_seoul['경도'] = zip(*df_real_estate_seoul['주소'].apply(get_lat_long))
df_real_estate_seoul.head()

{'documents': [{'address': {'address_name': '서울 종로구 청운동 1', 'b_code': '1111010100', 'h_code': '1111051500', 'main_address_no': '1', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '종로구', 'region_3depth_h_name': '청운효자동', 'region_3depth_name': '청운동', 'sub_address_no': '', 'x': '126.968196064077', 'y': '37.5912867191491'}, 'address_name': '서울 종로구 청운동 1', 'address_type': 'REGION_ADDR', 'road_address': {'address_name': '서울 종로구 자하문로36길 16-14', 'building_name': '청운벽산빌리지', 'main_building_no': '16', 'region_1depth_name': '서울', 'region_2depth_name': '종로구', 'region_3depth_name': '청운동', 'road_name': '자하문로36길', 'sub_building_no': '14', 'underground_yn': 'N', 'x': '126.968017264207', 'y': '37.5915828269406', 'zone_no': '03046'}, 'x': '126.968196064077', 'y': '37.5912867191491'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}
위도: 37.5912867191491, 경도: 126.968196064077
주소: 서울특별시 종로구 청운동 1, 위도: 37.5912867191491, 경도: 126.968196064077


,주소,단지명_공시가격,단지종류,세대수,구,위도,경도
0,서울특별시 종로구 청운동 1,청운벽산빌리지,2,126,종로구,37.5912867191491,126.968196064077
1,서울특별시 종로구 청운동 3,인텔빌라BC동,2,18,종로구,37.5910669705112,126.967546346289
2,서울특별시 종로구 청운동 3-150,인텔빌라A동,2,8,종로구,37.590602353175,126.967965259874
3,서울특별시 종로구 청운동 4-1,아델하우스,2,12,종로구,37.5905575566464,126.967245951136
4,서울특별시 종로구 청운동 4-3,GRACETUSCANII,2,4,종로구,37.590016555159,126.967734190892


In [457]:
# df_real_estate_seoul 파일로 저장하기
df_real_estate_seoul.to_csv('data/서울시_공동주택_위경도.csv', index=False, encoding='utf-8')

In [458]:
df_real_estate_seoul.info()

<class 'pandas.DataFrame'>
RangeIndex: 106097 entries, 0 to 106096
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   주소        106097 non-null  str  
 1   단지명_공시가격  106097 non-null  str  
 2   단지종류      106097 non-null  int64
 3   세대수       106097 non-null  int64
 4   구         106097 non-null  str  
 5   위도        99988 non-null   str  
 6   경도        99988 non-null   str  
dtypes: int64(2), str(5)
memory usage: 5.7 MB


In [459]:
# 위도와 경도가 null인 행들만 추출하여 데이터 프레임으로 만들고 내보내기(일일 호출 제한 때문에 일부 주소에서 위도와 경도 정보가 누락될 수 있음)
missing_lat_long = df_real_estate_seoul[df_real_estate_seoul['위도'].isnull() | df_real_estate_seoul['경도'].isnull()]
missing_lat_long.to_csv('data/위도경도_결측값.csv', index=False, encoding='utf-8')

일일 호출 제한으로 인해 누락된 위경도 값 채워넣기

In [463]:
import requests

url = "https://dapi.kakao.com/v2/local/search/address.json"
headers2 = {

    "Authorization": "KakaoAK KAKAO_API_KEY_2"

}

params = {
    "query": "서울특별시 강남구 테헤란로 212"
}

res = requests.get(url, headers=headers2, params=params)
print(res.json())

{'documents': [{'address': {'address_name': '서울 강남구 역삼동 718-5', 'b_code': '1168010100', 'h_code': '1168065000', 'main_address_no': '718', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_h_name': '역삼2동', 'region_3depth_name': '역삼동', 'sub_address_no': '5', 'x': '127.039600248343', 'y': '37.5012767241426'}, 'address_name': '서울 강남구 테헤란로 212', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '서울 강남구 테헤란로 212', 'building_name': '멀티캠퍼스', 'main_building_no': '212', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_name': '역삼동', 'road_name': '테헤란로', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.039600248343', 'y': '37.5012767241426', 'zone_no': '06220'}, 'x': '127.039600248343', 'y': '37.5012767241426'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}


In [464]:
# 카카오맵 API로 위도, 경도 정보 가져오기
params = {
    "query": missing_lat_long['주소'].iloc[0]
}
res = requests.get(url, headers=headers2, params=params)
print(res.json())
# 위도, 경도 정보 추출
if res.status_code == 200 and res.json().get('documents'):
    latitude = res.json()['documents'][0]['y']
    longitude = res.json()['documents'][0]['x']
    print(f"위도: {latitude}, 경도: {longitude}")
# 위도, 경도 정보 가져오기 함수 정의
def get_lat_long(address):
    params = {
        "query": address
    }
    res = requests.get(url, headers=headers, params=params)
    if res.status_code == 200 and res.json().get('documents'):
        latitude = res.json()['documents'][0]['y']
        longitude = res.json()['documents'][0]['x']
        return latitude, longitude
    else:
        return None, None
# 위도, 경도 정보 가져오기 함수 테스트
test_address = missing_lat_long['주소'].iloc[0]
lat, long = get_lat_long(test_address)
print(f"주소: {test_address}, 위도: {lat}, 경도: {long}")
# 위도, 경도 정보 가져오기 함수 적용
missing_lat_long['위도'], missing_lat_long['경도'] = zip(*missing_lat_long['주소'].apply(get_lat_long))
missing_lat_long.head()

{'documents': [], 'meta': {'is_end': True, 'pageable_count': 0, 'total_count': 0}}
주소: 서울특별시 중구 남산동2가 39-4, 위도: None, 경도: None


,주소,단지명_공시가격,단지종류,세대수,구,위도,경도
2502,서울특별시 중구 남산동2가 39-4,능마루빌라,3,11,중구,NaN,NaN
13313,서울특별시 동대문구 용두동 39-1,청량리역한양수자인그라시엘,1,1152,동대문구,NaN,NaN
63341,서울특별시 강서구 마곡동 725-1,마곡힐스테이트,1,603,강서구,NaN,NaN
66589,서울특별시 구로구 고척동 100-7,고척아이파크(주상복합),1,1459,구로구,NaN,NaN
69091,서울특별시 구로구 항동 가-238,하버라인4단지,1,297,구로구,NaN,NaN


In [465]:
missing_lat_long.to_csv('data/위도경도_결측값_채운거.csv', index=False, encoding='utf-8')

In [466]:
# df_real_estate_seoul에서 위도, 경도 정보가 null인 행들을 missing_lat_long의 위도, 경도 정보로 채워넣기
for idx, row in missing_lat_long.iterrows():
    df_real_estate_seoul.loc[df_real_estate_seoul['주소'] == row['주소'], ['위도', '경도']] = row[['위도', '경도']].values
df_real_estate_seoul.info()

<class 'pandas.DataFrame'>
RangeIndex: 106097 entries, 0 to 106096
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   주소        106097 non-null  str  
 1   단지명_공시가격  106097 non-null  str  
 2   단지종류      106097 non-null  int64
 3   세대수       106097 non-null  int64
 4   구         106097 non-null  str  
 5   위도        106086 non-null  str  
 6   경도        106086 non-null  str  
dtypes: int64(2), str(5)
memory usage: 5.7 MB


In [472]:
# 위도 경도 null 값 찾기
print(df_real_estate_seoul[df_real_estate_seoul['위도'].isnull() | df_real_estate_seoul['경도'].isnull()])

# 결측치 처리: 위도, 경도 둘 다 null인 경우는 제거
df_real_estate_seoul.dropna(subset=['위도', '경도'], inplace=True)
df_real_estate_seoul.info()

                          주소       단지명_공시가격  단지종류   세대수     구   위도   경도
2502     서울특별시 중구 남산동2가 39-4          능마루빌라     3    11    중구  NaN  NaN
13313    서울특별시 동대문구 용두동 39-1  청량리역한양수자인그라시엘     1  1152  동대문구  NaN  NaN
63341    서울특별시 강서구 마곡동 725-1        마곡힐스테이트     1   603   강서구  NaN  NaN
66589    서울특별시 구로구 고척동 100-7   고척아이파크(주상복합)     1  1459   구로구  NaN  NaN
69091     서울특별시 구로구 항동 가-238        하버라인4단지     1   297   구로구  NaN  NaN
69092     서울특별시 구로구 항동 가-239        하버라인9단지     1   298   구로구  NaN  NaN
69093     서울특별시 구로구 항동 가-240       하버라인10단지     1   297   구로구  NaN  NaN
92064      서울특별시 강남구 자곡동 BL-        디아크리온강남     1   597   강남구  NaN  NaN
92065      서울특별시 강남구 자곡동 BL      LH수서1단지아파트     1   830   강남구  NaN  NaN
101683  서울특별시 강동구 고덕동 BL-3-1      고덕풍경채어바니티     1   780   강동구  NaN  NaN
106096  서울특별시 강동구 강일동 산22-36      강동리엔파크9단지     1   366   강동구  NaN  NaN
<class 'pandas.DataFrame'>
Index: 106086 entries, 0 to 106095
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype
--- 

In [467]:
df_real_estate_seoul.head()

,주소,단지명_공시가격,단지종류,세대수,구,위도,경도
0,서울특별시 종로구 청운동 1,청운벽산빌리지,2,126,종로구,37.5912867191491,126.968196064077
1,서울특별시 종로구 청운동 3,인텔빌라BC동,2,18,종로구,37.5910669705112,126.967546346289
2,서울특별시 종로구 청운동 3-150,인텔빌라A동,2,8,종로구,37.590602353175,126.967965259874
3,서울특별시 종로구 청운동 4-1,아델하우스,2,12,종로구,37.5905575566464,126.967245951136
4,서울특별시 종로구 청운동 4-3,GRACETUSCANII,2,4,종로구,37.590016555159,126.967734190892


In [473]:
# df_real_estate_seoul 파일로 저장하기
df_real_estate_seoul.to_csv('data/서울시_공동주택_위경도_최종.csv', index=False, encoding='utf-8')